# Symmetry Phase 4 - Linear Decoding vs Rank

This notebook evaluates coordinate decoding performance as a function of
the number of PCA components used as features.

In [ ]:
import numpy as np

from boardGPT.analysis import (
    PolicyConfig,
    SyntheticGPTTrainConfig,
    SyntheticWorld,
    build_tiny_gpt_for_synthetic,
    evaluate_linear_decoding_vs_rank,
    extract_mean_token_representations_from_model,
    generate_synthetic_sequences,
    lattice_nd_concept,
    pca_from_gram,
    train_gpt_on_synthetic_sequences,
)

## 1) Build a 2D synthetic world and train a tiny GPT

In [ ]:
concept = lattice_nd_concept(shape=(6, 6), periodic=(False, False), name='grid2d')
world = SyntheticWorld(concept=concept)

data_cfg = PolicyConfig(
    length=40,
    num_sequences=1200,
    seed=1,
    transition_kernel='exponential',
    transition_scale=0.25,
    transition_strength=1.0,
    epsilon_uniform=0.04,
)
dataset = generate_synthetic_sequences(world, data_cfg)

model = build_tiny_gpt_for_synthetic(
    vocab_size=world.vocab_size,
    block_size=dataset.sequences.shape[1] - 1,
    n_layer=2,
    n_head=2,
    n_embd=64,
    use_flash=False,
)
train_cfg = SyntheticGPTTrainConfig(max_steps=100, batch_size=64, seed=1, device='cpu', log_interval=50)
_ = train_gpt_on_synthetic_sequences(model, dataset.sequences, train_cfg)

## 2) Token representations -> PCA feature bank

In [ ]:
reprs = extract_mean_token_representations_from_model(
    model=model,
    sequences=dataset.sequences,
    layer=1,
    vocab_size=world.vocab_size,
    device='cpu',
    batch_size=128,
)
gram = reprs @ reprs.T
coords, evals, _ = pca_from_gram(gram, rank=None, center=True)
print('feature bank shape:', coords.shape)

## 3) Decode latent 2D coordinates vs rank

In [ ]:
targets = concept.coordinates  # [V, 2]
ranks = np.arange(1, min(20, coords.shape[1]) + 1)

res_ridgeless = evaluate_linear_decoding_vs_rank(
    features=coords,
    targets=targets,
    ranks=ranks,
    num_trials=40,
    train_size=targets.shape[0] // 2,
    ridge_alpha=0.0,
    seed=0,
)

res_ridge = evaluate_linear_decoding_vs_rank(
    features=coords,
    targets=targets,
    ranks=ranks,
    num_trials=40,
    train_size=targets.shape[0] // 2,
    ridge_alpha=1e-2,
    seed=0,
)

print('Ridgeless test rel MSE first/last:', res_ridgeless.test_rel_mse_mean[0], res_ridgeless.test_rel_mse_mean[-1])
print('Ridge test rel MSE first/last    :', res_ridge.test_rel_mse_mean[0], res_ridge.test_rel_mse_mean[-1])

If you want to plot curves, you can add matplotlib in a follow-up cell and
draw train/test means with std bars.